# CXR-MultiQuant — Phase 2: Model Architecture
Building the Multimodal deep learning architecture using **TensorFlow / Keras**.

- **Vision**: DenseNet-121 (ImageNet pretrained)
- **Text**: BERT-base via TensorFlow Hub (same architecture as ClinicalBERT: 12 layers, 768 hidden)
- **Fusion**: Co-Attention (2× Multi-Head Attention)
- **Loss**: Focal Loss (handles class imbalance)

In [4]:
# Cell 1: Install Dependencies
# Using tensorflow-hub for BERT (no more TFAutoModel dependency issues!)
# 'transformers' is only used for the tokenizer (AutoTokenizer) — works in any version.
!pip install --quiet tensorflow-hub transformers datasets pandas scikit-learn keras-tuner

In [1]:
# Cell 2: Imports and Setup
import os
import io
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub
from tensorflow import keras
from tensorflow.keras import layers
from transformers import AutoTokenizer  # Only the tokenizer — works in ALL versions

from datasets import load_dataset
from PIL import Image

# Set seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print(f"TensorFlow Version: {tf.__version__}")
print("GPU Available:", tf.config.list_physical_devices('GPU'))
print("All imports successful!")


TensorFlow Version: 2.21.0
GPU Available: []
All imports successful!


In [2]:
# Cell 3: Load Data and Tokenizer
print("Loading Hugging Face dataset...")
hf_dataset = load_dataset("itsanmolgupta/mimic-cxr-dataset", split='train')
hf_df = hf_dataset.to_pandas()


print("Loading generated CSV labels...")
train_csv = pd.read_csv('train_labels.csv')
val_csv = pd.read_csv('val_labels.csv')
test_csv = pd.read_csv('test_labels.csv')


train_df = pd.merge(hf_df, train_csv[['impression', 'severity']], on='impression', how='inner')
val_df = pd.merge(hf_df, val_csv[['impression', 'severity']], on='impression', how='inner')
test_df = pd.merge(hf_df, test_csv[['impression', 'severity']], on='impression', how='inner')

print(f"Train size: {len(train_df)} | Val size: {len(val_df)} | Test size: {len(test_df)}")

# Load ClinicalBERT Tokenizer (tokenizer only — no model dependency issues)
tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
MAX_LEN = 512  # Max token length for the reports (Findings + Impression)


Loading Hugging Face dataset...
Loading generated CSV labels...
Train size: 2104814 | Val size: 263131 | Test size: 308584


c:\Users\BIT\anaconda3\envs\kill_env\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

c:\Users\BIT\anaconda3\envs\kill_env\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\BIT\.cache\huggingface\hub\models--emilyalsentzer--Bio_ClinicalBERT. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

In [ ]:
# Cell 4: Create tf.data.Dataset Generator
label_map = {'Mild': 0, 'Moderate': 1, 'Severe': 2}

def extract_image(img_data):
    if isinstance(img_data, dict) and 'bytes' in img_data:
        img = Image.open(io.BytesIO(img_data['bytes'])).convert('RGB')
    else:
        img = img_data.convert('RGB')
    
    img = img.resize((224, 224))
    return np.array(img) / 255.0  # Normalize to [0, 1]

def data_generator(df):
    for idx, row in df.iterrows():
        img_array = extract_image(row['image'])
        
        tokens = tokenizer(
            str(row['findings']) + " " + str(row['impression']),
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors="np"  # Return numpy arrays (compatible with tf.data)
        )
        input_ids = tokens['input_ids'].squeeze()
        attention_mask = tokens['attention_mask'].squeeze()
        
        label = label_map[row['severity']]
        yield (img_array, input_ids, attention_mask), label

output_signature = (
    (
        tf.TensorSpec(shape=(224, 224, 3), dtype=tf.float32),  # Image
        tf.TensorSpec(shape=(MAX_LEN,), dtype=tf.int32),       # Input IDs
        tf.TensorSpec(shape=(MAX_LEN,), dtype=tf.int32)        # Attention Mask
    ),
    tf.TensorSpec(shape=(), dtype=tf.int32)                    # Label
)

BATCH_SIZE = 16



train_ds = tf.data.Dataset.from_generator(lambda: data_generator(train_df), output_signature=output_signature)
train_ds = train_ds.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_generator(lambda: data_generator(val_df), output_signature=output_signature)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("tf.data Pipelines created successfully!")

tf.data Pipelines created successfully!


## 1. Image Encoder (DenseNet-121)
Pretrained on ImageNet, augmented, and projected to 256 dimensions.

In [ ]:
# Cell 5: Image Encoder
def create_image_encoder():
    image_input = keras.Input(shape=(224, 224, 3), name="image_input")
    
    # Data Augmentation
    x = layers.RandomRotation(0.1)(image_input)
    x = layers.RandomZoom(0.1)(x)
    
    # Pretrained DenseNet121
    densenet = keras.applications.DenseNet121(
        include_top=False,
        weights="imagenet",
        input_tensor=x
    )
    densenet.trainable = False  # Freeze for initial training
    
    x = densenet.output
    x = layers.GlobalAveragePooling2D()(x)
    
    # Dense Projection Layer -> 256-dim
    img_features = layers.Dense(256, activation='relu', name="img_projection")(x)
    
    return keras.Model(inputs=image_input, outputs=img_features, name="Image_Encoder")

image_encoder = create_image_encoder()
image_encoder.summary()

## 2. Text Encoder (BERT via TensorFlow Hub)
Uses BERT-base-uncased (same architecture as ClinicalBERT: 12 layers, 768 hidden dims, 12 attention heads).
Extracts the [CLS] pooled output and projects to 256 dimensions.

**Why TF Hub instead of HuggingFace?** HuggingFace `transformers` v5+ removed all TensorFlow model classes.
TF Hub is Google's own model repository — zero compatibility issues with TensorFlow.

In [ ]:
# Cell 6: Text Encoder (BERT via TensorFlow Hub)
BERT_HANDLE = "https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/4"

def create_text_encoder():
    input_ids = keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name="input_ids")
    attention_mask = keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name="attention_mask")
    
    # Create token_type_ids as zeros (single-sentence input)
    token_type_ids = tf.zeros_like(input_ids)
    
    # Load BERT from TensorFlow Hub (12 layers, 768 hidden, 12 heads — same as ClinicalBERT)
    bert_layer = hub.KerasLayer(BERT_HANDLE, trainable=False, name="bert_encoder")
    
    # TF Hub BERT expects this specific input dict format
    bert_output = bert_layer({
        "input_word_ids": input_ids,
        "input_mask": attention_mask,
        "input_type_ids": token_type_ids
    })
    
    # Extract [CLS] pooled output (768-dim)
    cls_token = bert_output["pooled_output"]
    
    # Dense Projection Layer -> 256-dim (as per ARCHITECTURE.md)
    txt_features = layers.Dense(256, activation='relu', name="txt_projection")(cls_token)
    
    return keras.Model(inputs=[input_ids, attention_mask], outputs=txt_features, name="Text_Encoder")

print("Downloading BERT model from TF Hub (first time only, ~400MB)...")
text_encoder = create_text_encoder()
text_encoder.summary()

## 3. Co-Attention Fusion & Classifier
Cross attention between Image and Text, followed by the severity classifier.

In [ ]:
# Cell 7: Full Multimodal Model with Co-Attention
def create_multimodal_model():
    # Inputs
    image_input = keras.Input(shape=(224, 224, 3), name="image_input")
    input_ids = keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name="input_ids")
    attention_mask = keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name="attention_mask")
    
    # Encoders
    img_feats = image_encoder(image_input)                 # (batch, 256)
    txt_feats = text_encoder([input_ids, attention_mask])  # (batch, 256)
    
    # To use MultiHeadAttention, we need 3D tensors: (batch_size, seq_len, dim)
    img_feats_seq = tf.expand_dims(img_feats, axis=1)      # (batch, 1, 256)
    txt_feats_seq = tf.expand_dims(txt_feats, axis=1)      # (batch, 1, 256)
    
    # --- CO-ATTENTION FUSION ---
    # Cross 1: Image -> Text (Q: Image, K/V: Text)
    attn_img_txt = layers.MultiHeadAttention(num_heads=8, key_dim=32, name="cross1_img_txt")(
        query=img_feats_seq, value=txt_feats_seq, key=txt_feats_seq
    )
    # Residual & LayerNorm
    norm1 = layers.LayerNormalization()(img_feats_seq + attn_img_txt)
    
    # Cross 2: Text -> Image (Q: Text, K/V: Image)
    attn_txt_img = layers.MultiHeadAttention(num_heads=8, key_dim=32, name="cross2_txt_img")(
        query=txt_feats_seq, value=img_feats_seq, key=img_feats_seq
    )
    # Residual & LayerNorm
    norm2 = layers.LayerNormalization()(txt_feats_seq + attn_txt_img)
    
    # Squeeze back to 2D and Concatenate -> 512-dim
    norm1_flat = tf.squeeze(norm1, axis=1)
    norm2_flat = tf.squeeze(norm2, axis=1)
    fused = layers.Concatenate(name="fused_concat")([norm1_flat, norm2_flat])
    
    # --- CLASSIFIER HEAD ---
    x = layers.Dense(256, activation='relu', name="fc1")(fused)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Dense(128, activation='relu', name="fc2")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    
    output = layers.Dense(3, activation='softmax', name="severity_prediction")(x)
    
    return keras.Model(inputs=[image_input, input_ids, attention_mask], outputs=output, name="CXR_MultiQuant")

multimodal_model = create_multimodal_model()
multimodal_model.summary()

## 4. Compile with Focal Loss

In [ ]:
# Cell 8: Define Focal Loss and Compile Model
class CategoricalFocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=0.25, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        # y_true is passed as integer labels from the dataset (sparse)
        # We need to one-hot encode them first
        y_true_one_hot = tf.one_hot(tf.cast(y_true, tf.int32), depth=3)
        
        y_pred = tf.clip_by_value(y_pred, keras.backend.epsilon(), 1.0 - keras.backend.epsilon())
        cross_entropy = -y_true_one_hot * tf.math.log(y_pred)
        loss = self.alpha * tf.math.pow(1 - y_pred, self.gamma) * cross_entropy
        return tf.reduce_sum(loss, axis=-1)

multimodal_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=CategoricalFocalLoss(gamma=2.0, alpha=0.25),
    metrics=['accuracy', keras.metrics.AUC(name='auc_roc', multi_label=True)]
)

print("Model compiled successfully with Focal Loss!")

In [ ]:
# Cell 9: Hyperparameter Tuning (Bayesian Optimization)
import keras_tuner as kt
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

def build_model(hp):
    # Rebuild the model structure for the tuner
    image_input = keras.Input(shape=(224, 224, 3), name="image_input")
    input_ids = keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name="input_ids")
    attention_mask = keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name="attention_mask")
    
    img_feats = image_encoder(image_input)
    txt_feats = text_encoder([input_ids, attention_mask])
    
    img_feats_seq = tf.expand_dims(img_feats, axis=1)
    txt_feats_seq = tf.expand_dims(txt_feats, axis=1)
    
    attn_img_txt = layers.MultiHeadAttention(num_heads=8, key_dim=32)(
        query=img_feats_seq, value=txt_feats_seq, key=txt_feats_seq
    )
    norm1 = layers.LayerNormalization()(img_feats_seq + attn_img_txt)
    
    attn_txt_img = layers.MultiHeadAttention(num_heads=8, key_dim=32)(
        query=txt_feats_seq, value=img_feats_seq, key=img_feats_seq
    )
    norm2 = layers.LayerNormalization()(txt_feats_seq + attn_txt_img)
    
    norm1_flat = tf.squeeze(norm1, axis=1)
    norm2_flat = tf.squeeze(norm2, axis=1)
    fused = layers.Concatenate()([norm1_flat, norm2_flat])
    
    # Hyperparameters for Dense and Dropout
    hp_dropout_1 = hp.Choice('dropout_1', values=[0.2, 0.3, 0.5])
    hp_dropout_2 = hp.Choice('dropout_2', values=[0.2, 0.3, 0.5])
    
    x = layers.Dense(256, activation='relu')(fused)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(hp_dropout_1)(x)
    
    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(hp_dropout_2)(x)
    
    output = layers.Dense(3, activation='softmax')(x)
    
    model = keras.Model(inputs=[image_input, input_ids, attention_mask], outputs=output)
    
    # Hyperparameters for Optimizer
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-3, 1e-4, 1e-5])
    hp_optimizer = hp.Choice('optimizer', values=['adam', 'rmsprop'])
    
    if hp_optimizer == 'adam':
        optimizer = keras.optimizers.Adam(learning_rate=hp_learning_rate)
    else:
        optimizer = keras.optimizers.RMSprop(learning_rate=hp_learning_rate)
        
    model.compile(
        optimizer=optimizer,
        loss=CategoricalFocalLoss(gamma=2.0, alpha=0.25),
        metrics=['accuracy', keras.metrics.AUC(name='auc_roc', multi_label=True)]
    )
    return model

tuner = kt.BayesianOptimization(
    build_model,
    objective=kt.Objective('val_auc_roc', direction='max'),
    max_trials=10,  # Bayesian Optimization trials
    directory='keras_tuner_dir',
    project_name='CXR_MultiQuant_Tuning'
)

tuner.search_space_summary()


In [ ]:
# Cell 10: Callbacks & Start Tuning
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('best_model.keras', save_best_only=True, monitor='val_loss'),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
]

# To start tuning, uncomment the line below:
# tuner.search(train_ds, validation_data=val_ds, epochs=30, callbacks=callbacks)

# To retrieve the best model after tuning:
# best_model = tuner.get_best_models(num_models=1)[0]
# best_model.summary()
